# 04. 学習済モデルで自動プレイ & スコア出力

`03_train_bc.ipynb` でダウンロードした `.pt` ファイルを `../models/checkpoints/` に配置してから実行してください。

In [ ]:
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace

ENV_ID = "SuperMarioBros-1-1-v0"

env = JoypadSpace(gym_super_mario_bros.make(ENV_ID, render_mode="human"), SIMPLE_MOVEMENT)

# Same decimation as 01_collect_data.ipynb. The training data updates its frame
# history at 30fps (once every 2 frames), so inference picks an action at the
# same rate and holds it in between (frame skip / action repeat).
TARGET_FPS = 30
RECORD_EVERY_N = round(env.metadata["render_fps"] / TARGET_FPS)

print("SIMPLE_MOVEMENT:", SIMPLE_MOVEMENT)
print("make sure this ordering matches the one used in 03_train_bc.ipynb")
print(f"render_fps={env.metadata['render_fps']} -> inference fps={TARGET_FPS} (RECORD_EVERY_N={RECORD_EVERY_N})")

In [ ]:
# --- Keep this block identical in 03_train_bc.ipynb and 04_playagent.ipynb ---
from collections import deque

import cv2
import numpy as np
import torch
import torch.nn as nn

FRAME_SIZE = (84, 84)
STACK_SIZE = 4
NUM_ACTIONS = len(SIMPLE_MOVEMENT)


def preprocess_frame(frame: np.ndarray) -> np.ndarray:
    """Convert an RGB frame to grayscale and resize it to 84x84."""
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    return cv2.resize(gray, FRAME_SIZE, interpolation=cv2.INTER_AREA)


class FrameStacker:
    """Stack the most recent STACK_SIZE preprocessed frames."""

    def __init__(self, stack_size: int = STACK_SIZE):
        self.stack_size = stack_size
        self.frames = deque(maxlen=stack_size)

    def reset(self, processed_frame: np.ndarray) -> np.ndarray:
        self.frames.clear()
        for _ in range(self.stack_size):
            self.frames.append(processed_frame)
        return self.get()

    def step(self, processed_frame: np.ndarray) -> np.ndarray:
        self.frames.append(processed_frame)
        return self.get()

    def get(self) -> np.ndarray:
        return np.stack(self.frames, axis=0)


class MarioCNN(nn.Module):
    def __init__(self, num_actions: int = NUM_ACTIONS, in_channels: int = STACK_SIZE):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, *FRAME_SIZE)
            n_flatten = self.features(dummy).shape[1]
        self.head = nn.Sequential(
            nn.Linear(n_flatten, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.float() / 255.0
        x = self.features(x)
        return self.head(x)
# --- end of shared block ---

In [ ]:
from pathlib import Path

# change this to the file you downloaded from 03_train_bc.ipynb
CHECKPOINT_PATH = Path("../models/checkpoints/mario_bc_YYYYMMDD_HHMMSS.pt")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MarioCNN().to(device)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()
print(f"loaded: {CHECKPOINT_PATH} (device={device})")

In [ ]:
import csv
import time
from datetime import datetime

EPISODES = 5

RESULTS_DIR = Path("../results")
EVAL_DIR = RESULTS_DIR / "eval_runs"
VIDEO_DIR = RESULTS_DIR / "videos"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
EVAL_CSV_PATH = EVAL_DIR / f"{RUN_TS}.csv"
EVAL_FIELDS = ["episode", "score", "x_pos_max", "flag_get", "steps", "video_path"]

fps = env.metadata.get("render_fps", 60)
frame_duration = 1 / fps

results = []

with open(EVAL_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    csv_writer = csv.DictWriter(f, fieldnames=EVAL_FIELDS)
    csv_writer.writeheader()

    for episode in range(EPISODES):
        obs, info = env.reset()
        stacker = FrameStacker()
        stack = stacker.reset(preprocess_frame(obs))

        video_path = VIDEO_DIR / f"{RUN_TS}_ep{episode:02d}.mp4"
        h, w = obs.shape[0], obs.shape[1]
        video_writer = cv2.VideoWriter(
            str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)
        )

        steps = 0
        terminated = truncated = False
        while not (terminated or truncated):
            stack_tensor = torch.from_numpy(stack).unsqueeze(0).to(device)
            with torch.no_grad():
                action = model(stack_tensor).argmax(dim=1).item()

            # Match the training data's decimation (one action every
            # RECORD_EVERY_N frames) by holding the action for that many frames
            for _ in range(RECORD_EVERY_N):
                t0 = time.time()

                obs, reward, terminated, truncated, info = env.step(action)
                steps += 1

                env.render()
                video_writer.write(cv2.cvtColor(obs, cv2.COLOR_RGB2BGR))

                elapsed = time.time() - t0
                if elapsed < frame_duration:
                    time.sleep(frame_duration - elapsed)

                if terminated or truncated:
                    break

            stack = stacker.step(preprocess_frame(obs))

        video_writer.release()

        row = {
            "episode": episode,
            "score": info["score"],
            "x_pos_max": info["x_pos_max"],
            "flag_get": info["flag_get"],
            "steps": steps,
            "video_path": str(video_path),
        }
        csv_writer.writerow(row)
        results.append(row)
        print(
            f"episode {episode}: score={row['score']} x_pos_max={row['x_pos_max']} "
            f"flag_get={row['flag_get']} steps={steps} video={video_path.name}"
        )

env.close()

## 結果の集計

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
print(f"\n=== results over {EPISODES} episodes ===")
print(f"mean score: {results_df['score'].mean():.1f}")
print(f"mean distance (x_pos_max): {results_df['x_pos_max'].mean():.1f}")
print(f"clear rate: {results_df['flag_get'].mean():.1%}")
print(f"best score: {results_df['score'].max()} (episode {int(results_df['score'].idxmax())})")
print(f"\nlog: {EVAL_CSV_PATH}")
print("videos:")
for p in results_df["video_path"]:
    print(" -", p)

results_df